In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [5]:
from birddog.tracker import (
    PageTracker,
    DynamoDBPageChangeLogTable,
    DynamoDBPageTrackerTable,
    SQLitePageChangeLogTable,
    SQLitePageTrackerTable,
    PageChangeLog,
    WikiDocTracker,
    )

from birddog.wiki import (
    page_label,
    get_recent_changes,
    lookup_namespace_id,
    get_recent_changes_v2,
    _api_url,
    )

In [60]:
import re

_DIGIT_SPLIT_RE = re.compile(r"\d+|\D+", re.UNICODE)

def split_part(s: str) -> list[str]:
    """
    Split a string into substrings such that each substring is
    either all decimal digits (0–9) or contains no decimal digits.

    Examples:
        "Фонд-12а"   -> ["Фонд-", "12", "а"]
        "abc123def"  -> ["abc", "123", "def"]
        "12-34"      -> ["12", "-", "34"]
        "№7bis"      -> ["№", "7", "bis"]
        ""           -> []
    """
    if not s:
        return []
    return _DIGIT_SPLIT_RE.findall(s)


In [61]:
def _make_sortable(s):
    if not s:
        return ""
    if s[0].isdigit():
        return f"{int(s):08d}"
    return s
    
def sortable_part(s):
    if not s:
        return ""
    s_parts = [_make_sortable(p) for p in split_part(s)]
    return "".join(s_parts)

In [64]:
def sortable_label(s):
    if not s:
        return ""
    parts = [sortable_part(p) for p in s.split("/")]
    return "/".join(parts)

In [3]:
tracker = PageTracker()

In [4]:
len(tracker._page_dict)

177747

In [6]:
titles = list(tracker._page_dict.keys())

In [7]:
titles[0]

'Архів:ДАЖО/173/1'

In [8]:
labels = [page_label(t) for t in titles]

In [9]:
labels[0]

'DAZHO-D/173/1'

In [41]:
labels[:10]

['DAZHO-D/173/1',
 'TSDIAL-_/134/1/1337',
 'DAKO-D/280/2/441',
 'TSDIAL-_/134/1',
 'TSDIAL-_/134/1/2',
 'DADNO-R/R-2276/1/186',
 'DADNO-R/R-2276/1/184',
 'DADNO-R/R-2276/1/1839',
 'DADNO-R/R-2276/1/1838',
 'DADNO-R/R-2276/1/1837']

In [42]:
parts = [p for l in labels if l for p in l.split("/")]

In [43]:
len(parts)

643640

In [44]:
parts = list(set(parts))

In [63]:
sortable_part(parts[1])

'R-00000019'

In [66]:
sortable_label(labels[1])

'TSDIAL-_/00000134/00000001/00001337'

In [45]:
split_label(parts[0])

['10', 'sch']

In [46]:
frags = [f for p in parts for f in split_label(p) ]

In [48]:
frags = list(set(frags))

In [49]:
num_strings = [s for s in frags if s[0].isdigit()]

In [50]:
num_strings[:10]

['4304', '5158', '1097', '5184', '4079', '241', '7349', '33', '64102', '6025']

In [51]:
max([int(s) for s in num_strings])

300017

In [52]:
s_strings = [s for s in frags if not s[0].isdigit()]

In [53]:
s_strings[:10]

['DAHMO-R',
 'DALUO-D',
 'DAKIRO-D',
 'DAPO-D',
 'DADNO-D',
 'NN',
 'RGADA-_',
 'a',
 'pr',
 'V']

In [54]:
max([len(s) for s in s_strings])

30

In [55]:
num0_strings = [f"{int(s):08d}" for s in num_strings]

In [56]:
num0_strings[:10]

['00004304',
 '00005158',
 '00001097',
 '00005184',
 '00004079',
 '00000241',
 '00007349',
 '00000033',
 '00064102',
 '00006025']